### Region of Interest (ROI) Cropping with Tissue Padding

In [1]:
import os
import cv2
import pandas as pd

image_dir = "../Dataset/JPEGImages"
mapping_csv_path = "../Outputs/image_label_mapping.csv"
output_crop_dir = "../Outputs/cropped_images"

# Create the output directory if doesn't exist
os.makedirs(output_crop_dir, exist_ok=True)


df = pd.read_csv(mapping_csv_path)

# 3. Configure preprocessing parameters
padding = 10        # Boundary safety buffer to catch margin tissue context
target_size = (224, 224)  # Standard dimensions to ensure a fair comparison with Member A

print("Beginning ROI extraction for 5,000 ultrasound images...")

# Loop through every single row inside the mapping spreadsheet
for idx, row in df.iterrows():
    img_name = row['image_name']
    xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
    
    # Construct absolute path to the raw image
    img_path = os.path.join(image_dir, img_name)
    
    # Read the image file using OpenCV
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Image file missing or unreadable: {img_name}. Skipping.")
        continue
        
    # Get the real height and width boundaries of the raw ultrasound frame
    height, width = img.shape[:2]
    
    # Apply padding while locking values between 0 and maximum image size
    # This prevents the script from crashing if a nodule sits right against the screen edge
    xmin_pad = max(0, xmin - padding)
    ymin_pad = max(0, ymin - padding)
    xmax_pad = min(width, xmax + padding)
    ymax_pad = min(height, ymax + padding)
    
    # Extract the isolated ROI matrix slice
    # CRITICAL COMPUTER VISION NOTE: OpenCV reads coordinates as [rows, columns]
    # This means you must index your crop as [ymin:ymax, xmin:xmax]
    cropped_roi = img[ymin_pad:ymax_pad, xmin_pad:xmax_pad]
    
    # Resize the nodule slice to standard dimensions 
    # (Ensures all feature maps scale consistently regardless of organic tumor size)
    resized_roi = cv2.resize(cropped_roi, target_size)
    
    # Construct save path and export the final cropped file
    save_path = os.path.join(output_crop_dir, img_name)
    cv2.imwrite(save_path, resized_roi)

print(f"\nSuccess! All 5,000 images cropped, padded, standardized, and saved to: {output_crop_dir}")

ModuleNotFoundError: No module named 'cv2'